# Chapter 9: From Predictions to Potential Profit

This companion notebook now uses the real Football-Data.co.uk Premier League file `E0.csv` for the 2025-26 season, matching the rewritten Chapter 9 text. The workflow below stays intentionally compact: load the real data, inspect the minimal columns, evaluate baseline strategies, fit a simple calibrated model on the first two-thirds of the season, and assess flat-stake and Kelly betting on the final third.

## What This Notebook Covers

- loading the real `E0_2025_2026.csv` dataset
- previewing the minimal chapter-9 columns
- measuring home-team and favorite baselines
- fitting a simple calibrated multinomial logistic regression model
- evaluating flat-stake value betting and Kelly sizing on a chronological holdout set
- regenerating the figures used in the book section


In [ ]:
import importlib.util
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next(path for path in [cwd, *cwd.parents] if (path / 'Companion-Code').exists())
MODULE_PATH = ROOT / 'Companion-Code' / 'extras' / 'chapter-9' / 'real_data_betting_workflow.py'
spec = importlib.util.spec_from_file_location('chapter9_real', MODULE_PATH)
chapter9_real = importlib.util.module_from_spec(spec)
spec.loader.exec_module(chapter9_real)

matches = chapter9_real.load_matches()
train_df, valid_df = chapter9_real.train_validation_split(matches)
base_model, calibrated_model = chapter9_real.build_calibrated_model(train_df)
baselines = chapter9_real.baseline_paths(matches)
model_summary = chapter9_real.evaluate_model(base_model, calibrated_model, valid_df)

print(f'Matches loaded: {len(matches)}')
print(f'Train matches: {len(train_df)} | Validation matches: {len(valid_df)}')


## Minimal Dataset Preview

In [ ]:
preview = matches[chapter9_real.MIN_COLUMNS].head(10).copy()
preview


## Baseline Strategies

These are the same simple benchmarks used in the chapter: always back the home team, or always back the shortest Bet365 price.

In [ ]:
import pandas as pd

baseline_summary = pd.DataFrame({
    'Always Home': {
        'Accuracy': baselines['home']['accuracy'],
        'Profit': baselines['home']['profit'],
        'ROI': baselines['home']['roi'],
    },
    'Always Favorite': {
        'Accuracy': baselines['favorite']['accuracy'],
        'Profit': baselines['favorite']['profit'],
        'ROI': baselines['favorite']['roi'],
    },
}).T
baseline_summary


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
axes[0].plot(matches['Kickoff'], baselines['home']['roi_path'], color='#0B5D7A', linewidth=2)
axes[0].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[0].set_title('Home-Team ROI Over Time')
axes[0].set_ylabel('ROI')
axes[0].grid(alpha=0.25)
axes[1].plot(matches['Kickoff'], baselines['favorite']['roi_path'], color='#B63A1B', linewidth=2)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Favorite ROI Over Time')
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Calibrated Model on a Chronological Split

In [ ]:
model_metrics = pd.Series({
    'Base Accuracy': model_summary['base_accuracy'],
    'Base Log Loss': model_summary['base_log_loss'],
    'Calibrated Accuracy': model_summary['calibrated_accuracy'],
    'Calibrated Log Loss': model_summary['calibrated_log_loss'],
})
model_metrics


In [ ]:
chapter9_real.plot_calibration(
    valid_df,
    model_summary['calibrated_probabilities'],
    model_summary['classes'],
    ROOT / 'Book' / 'images' / 'ch09_model_calibration_actual.png',
)
plt.figure(figsize=(12, 4))
img = plt.imread(ROOT / 'Book' / 'images' / 'ch09_model_calibration_actual.png')
plt.imshow(img)
plt.axis('off')
plt.show()


## Value Betting and Kelly on the Validation Block

In [ ]:
betting_metrics = pd.Series({
    'Flat Value Bets': model_summary['flat_value_bets'],
    'Flat Value Win Rate': model_summary['flat_value_win_rate'],
    'Flat Value Profit': model_summary['flat_value_profit'],
    'Flat Value ROI': model_summary['flat_value_roi'],
    'Kelly Bets': model_summary['kelly_bets'],
    'Kelly Final Bankroll': model_summary['kelly_final_bankroll'],
    'Kelly Profit': model_summary['kelly_profit'],
    'Kelly ROI': model_summary['kelly_roi'],
})
betting_metrics


In [ ]:
chapter9_real.plot_model_roi(
    valid_df['Kickoff'],
    model_summary['flat_roi_path'],
    model_summary['kelly_roi_path'],
    ROOT / 'Book' / 'images' / 'ch09_model_value_roi_actual.png',
)
plt.figure(figsize=(12, 4))
img = plt.imread(ROOT / 'Book' / 'images' / 'ch09_model_value_roi_actual.png')
plt.imshow(img)
plt.axis('off')
plt.show()


## Reproducible Outputs

The helper script also writes a compact preview CSV and a summary JSON to `Companion-Code/extras/chapter-9/results/`. If you want to regenerate every figure used in the book from one command, run the script directly:

In [ ]:
print('python Companion-Code/extras/chapter-9/real_data_betting_workflow.py')
